In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    while current != current.parent:
        if (current / 'src' / 'models' / 'adapter.py').exists():
            return current
        current = current.parent
    raise RuntimeError('Could not locate DriftDetect repo root from the current working directory.')


REPO_ROOT = find_repo_root()
RESULTS_PATH = REPO_ROOT / 'results' / 'tables' / 'decoder_d1_results.npz'
FIGURE_DIR = REPO_ROOT / 'results' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PLOT_START = 10
PLOT_END = 189
PLOT_SLICE = slice(PLOT_START, PLOT_END + 1)
SUMMARY_STEPS = [25, 50, 100, 150, 175]

with np.load(RESULTS_PATH) as data:
    trained_decoder_drift = data['trained_decoder_drift']
    random_decoder_drift = data['random_decoder_drift']
    seeds = data['seeds']

n_seeds, horizon = trained_decoder_drift.shape
steps = np.arange(horizon)
steps_plot = steps[PLOT_SLICE]

trained_mean = trained_decoder_drift.mean(axis=0)
random_mean = random_decoder_drift.mean(axis=0)
trained_std = trained_decoder_drift.std(axis=0, ddof=1)
random_std = random_decoder_drift.std(axis=0, ddof=1)
trained_ci = 1.96 * trained_std / np.sqrt(n_seeds)
random_ci = 1.96 * random_std / np.sqrt(n_seeds)

ratio = np.divide(
    trained_mean,
    random_mean,
    out=np.full_like(trained_mean, np.nan),
    where=random_mean > 0,
)

print(f'Loaded {RESULTS_PATH.relative_to(REPO_ROOT)}')
print(f'trained_decoder_drift shape: {trained_decoder_drift.shape}')
print(f'random_decoder_drift shape: {random_decoder_drift.shape}')
print(f'seeds: {seeds[0]}-{seeds[-1]}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

trained_color = '#2563eb'
random_color = '#6b7280'

ax.plot(
    steps_plot,
    trained_mean[PLOT_SLICE],
    color=trained_color,
    linewidth=2.0,
    label='Trained decoder',
)
ax.fill_between(
    steps_plot,
    trained_mean[PLOT_SLICE] - trained_ci[PLOT_SLICE],
    trained_mean[PLOT_SLICE] + trained_ci[PLOT_SLICE],
    color=trained_color,
    alpha=0.20,
    linewidth=0,
)

ax.plot(
    steps_plot,
    random_mean[PLOT_SLICE],
    color=random_color,
    linewidth=2.0,
    label='Random decoder',
)
ax.fill_between(
    steps_plot,
    random_mean[PLOT_SLICE] - random_ci[PLOT_SLICE],
    random_mean[PLOT_SLICE] + random_ci[PLOT_SLICE],
    color=random_color,
    alpha=0.20,
    linewidth=0,
)

ax.set_xlim(PLOT_START, PLOT_END)
ax.set_xlabel('Imagination Horizon Step')
ax.set_ylabel('L2 Distance')
ax.set_title('D1: Trained vs Random Decoder Drift (Cheetah Run)')
ax.legend(frameon=True)
ax.grid(True, alpha=0.25, linewidth=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
drift_fig_path = FIGURE_DIR / 'd1_drift_comparison.pdf'
fig.savefig(drift_fig_path, bbox_inches='tight')
plt.show()
print(f'Saved figure: {drift_fig_path.relative_to(REPO_ROOT)}')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), dpi=300)

ax.plot(
    steps_plot,
    ratio[PLOT_SLICE],
    color='#ea580c',
    linewidth=2.0,
)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1.2, alpha=0.8)

ax.set_xlim(PLOT_START, PLOT_END)
ax.set_xlabel('Imagination Horizon Step')
ax.set_ylabel('Amplification Ratio (trained / random)')
ax.set_title('Decoder Amplification Ratio Over Horizon')
ax.grid(True, alpha=0.25, linewidth=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
ratio_fig_path = FIGURE_DIR / 'd1_amplification_ratio.pdf'
fig.savefig(ratio_fig_path, bbox_inches='tight')
plt.show()
print(f'Saved figure: {ratio_fig_path.relative_to(REPO_ROOT)}')


In [ ]:
print('D1 summary statistics')
print()
print(
    f"{'Step':>6}  {'Trained mean ± std':>24}  {'Random mean ± std':>24}  {'Ratio':>10}"
)
for step in SUMMARY_STEPS:
    trained_step_mean = trained_mean[step]
    random_step_mean = random_mean[step]
    trained_step_std = trained_std[step]
    random_step_std = random_std[step]
    step_ratio = ratio[step]
    print(
        f"{step:>6}  "
        f"{trained_step_mean:>10.3f} ± {trained_step_std:<8.3f}  "
        f"{random_step_mean:>10.3f} ± {random_step_std:<8.3f}  "
        f"{step_ratio:>10.3f}"
    )

overall_mean_ratio = np.nanmean(ratio[PLOT_SLICE])
print()
print(f'Overall mean ratio across steps [{PLOT_START}, {PLOT_END}]: {overall_mean_ratio:.3f}')


## D1 Result Summary

- Trained decoder shows higher drift than random: [YES / NO / FILL BASED ON PLOTS]
- Amplification ratio grows with horizon: [YES / NO / FILL BASED ON PLOTS]
- Preliminary interpretation: [FILL BASED ON PLOTS]. This [supports / contradicts / partially supports] the decoder amplification hypothesis.
- Caveat: D1 shows aggregate L2 amplification, but it does not yet confirm that `dc_trend` specifically is the component being amplified.
- Next step: D2 (Lipschitz probe) and frequency-band-specific analysis are needed to test whether the trained decoder preferentially amplifies low-frequency or `dc_trend` drift.
